# Linear model forecasting next-day returns

In [ ]:
import numpy as np
import pandas as pd

def synthetic_prices():
    rng = np.random.default_rng(7)
    returns = rng.normal(0.0005, 0.012, size=(120, 4))
    return pd.DataFrame(100.0 * (1.0 + returns).cumprod(axis=0), columns=list("ABCD"))


In [ ]:
from sklearn.linear_model import LinearRegression

def signal(prices):
    """Fit one pooled linear model on lagged returns; trade the sign of its forecast."""
    rets = prices.pct_change()
    X = pd.concat({"lag1": rets.shift(1).stack(), "lag5": rets.rolling(5).mean().shift(1).stack()}, axis=1)
    y = rets.stack().reindex(X.index)
    data = pd.concat([X, y.rename("y")], axis=1).dropna()
    train = data[data.index.get_level_values(0) < 40]
    model = LinearRegression().fit(train[["lag1", "lag5"]], train["y"])
    pred = pd.Series(model.predict(X.fillna(0.0)), index=X.index).unstack()
    pred.loc[pred.index < 40] = 0.0
    return np.sign(pred).reindex(index=prices.index, columns=prices.columns).fillna(0.0)


In [ ]:
def backtest(prices, cost_bps=10):
    positions = signal(prices)
    asset_returns = prices.pct_change().fillna(0.0)
    gross = (positions * asset_returns).mean(axis=1)
    turnover = positions.diff().abs().sum(axis=1).fillna(0.0)
    return gross - turnover * cost_bps / 10_000

def sharpe(pnl):
    pnl_std = pnl.std()
    return float(pnl.mean() / pnl_std * np.sqrt(252)) if pnl_std > 0 else 0.0

print(round(sharpe(backtest(synthetic_prices())), 3))
